In [ ]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [ ]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-2052ffe3-83d0-709d-c576-0b5d5d2ca619)


In [ ]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [ ]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')



Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


In [ ]:
# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/brain_signal_lfp')
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']


In [ ]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
list_dict.keys()

dict_keys(['brain_region_index', 'brain_region_index_Cosmos', 'brain_region_atlas', 'subject_list', 'exp_list', 'key_list', 'coordinate_list', 'depth_list', 'volume_list', 'brain_signal_lfp', 'brain_signal_ap', 'train_list_key', 'test_list_key', 'train_list_subject', 'test_list_subject', 'train_list_exp', 'test_list_exp', 'train_list_trial', 'test_list_trial', 'train_list_intest', 'test_list_intest', 'acronym_selec_list', 'valid_list_intest'])

In [ ]:
key = 'stimOff_times'
model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen'
subject_od_ind = torch.load(model_dir + f'/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
subject_od_list = torch.load(model_dir + f'/subject_od_list_Allen_{key}{0}.pt', weights_only=False)
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)
save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static'

In [ ]:
def iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, coordinate_list):
    data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index[train_ind], coordinate_list[train_ind])
    data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index[valid_ind], coordinate_list[valid_ind])
    data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index[test_ind], coordinate_list[test_ind])

    train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
    valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
    test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

    return train_iter, valid_iter, test_iter

In [ ]:
train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, coordinate_list)

In [ ]:
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':10,
    'save_dir':save_dir,
}


for x_train1, y_train, coordinate_train in train_iter_AnyNet:
    x_train = lfp_spectro(x_train1, spectro_args, train_args)
    break

In [ ]:
x_train.mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).size()

torch.Size([128, 1, 224, 28])

In [ ]:
# @title Training function
def net_train_AnyNet_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                if epoch0 == 0 and epoch == 0:
                    if accu_fun(py_train, y_train) == (torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu():
                        print('accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                    else:
                        print('accu_fun is wrong>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                        return
                print(accu_fun(py_train, y_train))
            # print(y_train.size())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append(accu_fun(py_train, y_train))
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).to(device))
            del x_valid, x_valid1
            acu_valid.append(accu_fun(py_valid, y_valid))
            # acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/AnyNet_L_Allen_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/AnyNet_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(acu_array_train, train_args['save_dir'] + f'/Model/AnyNet_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(acu_array_valid, train_args['save_dir'] + f'/Model/AnyNet_L_Allen_{key}_valid_acu{ind}.pt')

    return


def net_train_ViT_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_valid_acu{ind}.pt')
    torch.save(Classifier, train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_final_{ind}.pth')
    torch.save(epoch, train_args['save_dir'] + f'/Model/ViT_L_Allen_{key}_final_epoch{ind}.pt')

    return

def net_train_RNN_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        time0_train = time.time()
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).squeeze(1).permute(0, 2, 1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).squeeze(1).permute(0, 2, 1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/RNN_L_Allen_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/RNN_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/RNN_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/RNN_L_Allen_{key}_valid_acu{ind}.pt')

    return

def net_train_LC_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).squeeze(1).flatten(start_dim=1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).mean(dim=3, keepdim=True).repeat(1, 1, 1, 28).squeeze(1).flatten(start_dim=1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        # if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
        if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
            torch.save(Classifier, train_args['save_dir'] + f'/Model/LC_L_Allen_{key}_{ind}.pth')
            torch.save(epoch, train_args['save_dir'] + f'/Model/LC_L_Allen_{key}_epoch{ind}.pt')
            acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)


    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/LC_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/LC_L_Allen_{key}_valid_acu{ind}.pt')

    torch.save(time_array_train, train_args['save_dir'] + f'/Model/LC_L_time_array_train_{key}_{ind}.pt')
    torch.save(time_array_valid, train_args['save_dir'] + f'/Model/LC_L_time_array_valid_{key}_{ind}.pt')

    return


In [ ]:
train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
# train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_ViT, coordinate_list)
# train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_RNN, coordinate_list)

In [ ]:
# @title AnyNet model definition
c0 = 64 * 4
k0 = 1.0

model_args = {
    'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
    'stem_channels':c0,
}
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':50,
    'save_dir':save_dir,
}



In [ ]:
len(np.unique(communities_label))

32

In [ ]:
for ind in range(0, 1):
    Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(acronym_list)).to(device)
    Classifier.apply(init_cnn)
    net_train_AnyNet_L(train_iter_AnyNet, valid_iter_AnyNet, Classifier.to(device), spectro_args, train_args, key, ind, device)

accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>
tensor(0.)
tensor(0.1172)
tensor(0.1250)
tensor(0.1719)
tensor(0.1719)
tensor(0.1484)
tensor(0.2891)
tensor(0.2578)
tensor(0.2969)
tensor(0.3281)
tensor(0.2891)
tensor(0.2734)
tensor(0.3594)
Acu Train: 0.21773694455623627
Acu valid: 0.24931035935878754
tensor(0.2812)
tensor(0.3203)
tensor(0.3047)
tensor(0.3125)
tensor(0.3672)
tensor(0.3906)
tensor(0.2734)
tensor(0.4219)
tensor(0.2969)
tensor(0.3594)
tensor(0.3516)
tensor(0.3516)
tensor(0.3906)
Acu Train: 0.3551163673400879
Acu valid: 0.3181239366531372
tensor(0.4297)
tensor(0.4219)
tensor(0.4297)
tensor(0.2969)
tensor(0.4531)
tensor(0.4219)
tensor(0.4219)
tensor(0.4219)
tensor(0.3984)
tensor(0.5000)
tensor(0.3672)
tensor(0.4141)
tensor(0.4297)
Acu Train: 0.41751688718795776
Acu valid: 0.3272811770439148
tensor(0.3594)
tensor(0.4375)
tensor(0.4766)
tensor(0.3672)
tensor(0.5625)
tensor(0.4688)
tensor(0.4375)
tensor(0.4688)
tensor(0.4531)
tensor(0.4766)
tensor(0.5312)
tensor(0.4766)
tensor(0

In [ ]:
# del train_iter_ViT, valid_iter_ViT, test_iter_ViT

In [ ]:
# del train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet
train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, coordinate_list)

In [ ]:
# train_args['epochs'] = 50
for ind in range(0, 1):
    img_size, patch_size = (224, 28), (28, 28)
    num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
    emb_dropout, blk_dropout = 0.1, 0.1
    Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(acronym_list)).to(device)
    net_train_ViT_L(train_iter_ViT, valid_iter_ViT, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.0469)
tensor(0.0391)
tensor(0.0703)
tensor(0.1094)
tensor(0.0781)
tensor(0.0703)
tensor(0.1016)
tensor(0.1562)
tensor(0.1094)
tensor(0.0703)
tensor(0.1250)
tensor(0.1328)
Acu Train: 0.09075605869293213
Acu valid: 0.1311819851398468
tensor(0.1172)
tensor(0.1406)
tensor(0.1484)
tensor(0.1328)
tensor(0.1016)
tensor(0.1328)
tensor(0.1484)
tensor(0.1641)
tensor(0.2188)
tensor(0.1719)
tensor(0.2031)
tensor(0.2109)
tensor(0.2500)
Acu Train: 0.15914517641067505
Acu valid: 0.19306588172912598
tensor(0.1406)
tensor(0.2734)
tensor(0.2031)
tensor(0.2188)
tensor(0.2422)
tensor(0.1953)
tensor(0.2344)
tensor(0.2188)
tensor(0.2031)
tensor(0.2031)
tensor(0.2266)
tensor(0.2188)
tensor(0.2734)
Acu Train: 0.2220166027545929
Acu valid: 0.2492261528968811
tensor(0.2422)
tensor(0.2266)
tensor(0.2422)
tensor(0.2344)
tensor(0.2500)
tensor(0.2266)
tensor(0.2500)
tensor(0.2656)
tensor(0.3281)
tensor(0.2422)
tensor(0.2578)
tensor(0.2734)
tensor(0.1953)
Acu Train: 0.25983357429504395
Acu valid:

In [ ]:
# del train_iter_ViT, valid_iter_ViT, test_iter_ViT
train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, coordinate_list)

In [ ]:
RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(acronym_list),
    'data_len':28,
}
for ind in range(0, 1):
    Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
    net_train_RNN_L(train_iter_RNN, valid_iter_RNN, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.0625)
tensor(0.0625)
tensor(0.1016)
tensor(0.1719)
tensor(0.1875)
tensor(0.1562)
tensor(0.1719)
tensor(0.2422)
tensor(0.1797)
tensor(0.2578)
tensor(0.2266)
tensor(0.2734)
Acu Train: 0.17539723217487335
Acu valid: 0.2408071607351303
tensor(0.2812)
tensor(0.3594)
tensor(0.3047)
tensor(0.2812)
tensor(0.1797)
tensor(0.2500)
tensor(0.3750)
tensor(0.2891)
tensor(0.3594)
tensor(0.3281)
tensor(0.2891)
tensor(0.2969)
tensor(0.3750)
Acu Train: 0.3013070821762085
Acu valid: 0.26957693696022034
tensor(0.3594)
tensor(0.3359)
tensor(0.4062)
tensor(0.3359)
tensor(0.2812)
tensor(0.3438)
tensor(0.3828)
tensor(0.3594)
tensor(0.3438)
tensor(0.4453)
tensor(0.3672)
tensor(0.3906)
tensor(0.3359)
Acu Train: 0.3600027561187744
Acu valid: 0.2915053367614746
tensor(0.3984)
tensor(0.4062)
tensor(0.4141)
tensor(0.4141)
tensor(0.4531)
tensor(0.3281)
tensor(0.4297)
tensor(0.4766)
tensor(0.4375)
tensor(0.3594)
tensor(0.3750)
tensor(0.4688)
tensor(0.4531)
Acu Train: 0.4061354696750641
Acu valid: 0

In [ ]:
train_iter, valid_iter, test_iter = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, coordinate_list)

In [ ]:
LC_args = {
    'category_num':len(acronym_list),
}
Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)

torch.save(Classifier, train_args['save_dir'] + f'/Model/LC_L_Allen_chance_{key}_0.pth')

In [ ]:
for ind in range(0, 1):
    Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)
    net_train_LC_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.0469)
tensor(0.0625)
tensor(0.0391)
tensor(0.0547)
tensor(0.0312)
tensor(0.0625)
tensor(0.0391)
tensor(0.0156)
tensor(0.0547)
tensor(0.0703)
tensor(0.0391)
tensor(0.0625)
Acu Train: 0.051285870373249054
Acu valid: 0.057324785739183426
tensor(0.0859)
tensor(0.1094)
tensor(0.0547)
tensor(0.0625)
tensor(0.0547)
tensor(0.0391)
tensor(0.1016)
tensor(0.0703)
tensor(0.0938)
tensor(0.0625)
tensor(0.0547)
tensor(0.1172)
tensor(0.0312)
Acu Train: 0.0665239468216896
Acu valid: 0.07037072628736496
tensor(0.0391)
tensor(0.0781)
tensor(0.0781)
tensor(0.0938)
tensor(0.0625)
tensor(0.1016)
tensor(0.0469)
tensor(0.0703)
tensor(0.0625)
tensor(0.0781)
tensor(0.0469)
tensor(0.1094)
tensor(0.0547)
Acu Train: 0.0750451385974884
Acu valid: 0.0774700865149498
tensor(0.0781)
tensor(0.0859)
tensor(0.0391)
tensor(0.0625)
tensor(0.1484)
tensor(0.0781)
tensor(0.0391)
tensor(0.0938)
tensor(0.1172)
tensor(0.0859)
tensor(0.0938)
tensor(0.0469)
tensor(0.0781)
Acu Train: 0.08043423295021057
Acu vali

In [ ]:
from google.colab import runtime
runtime.unassign()